In [1]:
from pathlib import Path

import cv2
import numpy as np

In [12]:
DATA = "cards6"
SEQUENCE_DIR = Path(f"../data/train/HSI-RedNIR-FalseColor/{DATA}/")
GROUND_TRUTH_PATH = SEQUENCE_DIR / "groundtruth_rect.txt"

OUTPUT_VIDEO_PATH = Path(f"results/csrt_result_{DATA}.mp4")
OUTPUT_PREDICTIONS_PATH = Path(f"results/csrt_predictions_{DATA}.txt")

In [3]:
def load_ground_truth(path: Path) -> np.ndarray:
    """Load tab- or space-separated x, y, width, height boxes."""
    boxes = np.loadtxt(path, delimiter=None, dtype=np.float32)
    boxes = np.atleast_2d(boxes)

    if boxes.shape[1] != 4:
        raise ValueError(
            f"Expected four values per row, but found shape {boxes.shape}"
        )

    return boxes

In [4]:
def create_csrt_tracker():
    """
    Create a CSRT tracker while supporting different OpenCV Python layouts.
    """
    if hasattr(cv2, "TrackerCSRT_create"):
        return cv2.TrackerCSRT_create()

    if hasattr(cv2, "legacy") and hasattr(cv2.legacy, "TrackerCSRT_create"):
        return cv2.legacy.TrackerCSRT_create()

    raise RuntimeError(
        "CSRT is unavailable. Install opencv-contrib-python and make sure "
        "opencv-python is not installed in the same environment."
    )


In [5]:
def calculate_iou(box_a, box_b) -> float:
    """
    Calculate intersection over union.

    Both boxes use:
        x, y, width, height
    """
    ax, ay, aw, ah = map(float, box_a)
    bx, by, bw, bh = map(float, box_b)

    ax2 = ax + aw
    ay2 = ay + ah
    bx2 = bx + bw
    by2 = by + bh

    intersection_left = max(ax, bx)
    intersection_top = max(ay, by)
    intersection_right = min(ax2, bx2)
    intersection_bottom = min(ay2, by2)

    intersection_width = max(0.0, intersection_right - intersection_left)
    intersection_height = max(0.0, intersection_bottom - intersection_top)
    intersection_area = intersection_width * intersection_height

    area_a = max(0.0, aw) * max(0.0, ah)
    area_b = max(0.0, bw) * max(0.0, bh)

    union_area = area_a + area_b - intersection_area

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area

In [6]:
def draw_box(
    frame: np.ndarray,
    box,
    colour,
    label: str,
) -> None:
    x, y, width, height = box

    x = round(x)
    y = round(y)
    width = round(width)
    height = round(height)

    cv2.rectangle(
        frame,
        (x, y),
        (x + width, y + height),
        colour,
        2,
    )

    cv2.putText(
        frame,
        label,
        (x, max(20, y - 8)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.55,
        colour,
        2,
    )

In [7]:
def calculate_iou_xywh(
    predicted_box,
    ground_truth_box,
) -> float:
    """
    Calculate IoU for boxes in:
        x, y, width, height

    x and y represent the top-left corner.
    """
    pred_x, pred_y, pred_w, pred_h = map(float, predicted_box)
    gt_x, gt_y, gt_w, gt_h = map(float, ground_truth_box)

    if pred_w <= 0 or pred_h <= 0:
        return 0.0

    if gt_w <= 0 or gt_h <= 0:
        return 0.0

    pred_x2 = pred_x + pred_w
    pred_y2 = pred_y + pred_h

    gt_x2 = gt_x + gt_w
    gt_y2 = gt_y + gt_h

    intersection_x1 = max(pred_x, gt_x)
    intersection_y1 = max(pred_y, gt_y)
    intersection_x2 = min(pred_x2, gt_x2)
    intersection_y2 = min(pred_y2, gt_y2)

    intersection_width = max(
        0.0,
        intersection_x2 - intersection_x1,
    )
    intersection_height = max(
        0.0,
        intersection_y2 - intersection_y1,
    )

    intersection_area = (
        intersection_width * intersection_height
    )

    predicted_area = pred_w * pred_h
    ground_truth_area = gt_w * gt_h

    union_area = (
        predicted_area
        + ground_truth_area
        - intersection_area
    )

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area


def calculate_center_error(
    predicted_box,
    ground_truth_box,
) -> float:
    """
    Calculate Euclidean distance between the centres
    of two x, y, width, height boxes.
    """
    pred_x, pred_y, pred_w, pred_h = map(float, predicted_box)
    gt_x, gt_y, gt_w, gt_h = map(float, ground_truth_box)

    predicted_center_x = pred_x + pred_w / 2.0
    predicted_center_y = pred_y + pred_h / 2.0

    ground_truth_center_x = gt_x + gt_w / 2.0
    ground_truth_center_y = gt_y + gt_h / 2.0

    return float(
        np.sqrt(
            (predicted_center_x - ground_truth_center_x) ** 2
            + (predicted_center_y - ground_truth_center_y) ** 2
        )
    )


def calculate_competition_metrics(
    predictions,
    ground_truth,
) -> dict:
    """
    Calculate the HOT competition tracking metrics:

    1. Success plot AUC
    2. Distance Precision at 20 pixels, DP@20
    3. Mean IoU as an additional diagnostic

    The standard tracking success plot uses 21 overlap
    thresholds:
        0.00, 0.05, ..., 1.00
    """
    predictions = np.asarray(
        predictions,
        dtype=np.float64,
    )

    ground_truth = np.asarray(
        ground_truth,
        dtype=np.float64,
    )

    if predictions.ndim != 2 or predictions.shape[1] != 4:
        raise ValueError(
            f"Predictions must have shape (N, 4), "
            f"but got {predictions.shape}"
        )

    if ground_truth.ndim != 2 or ground_truth.shape[1] != 4:
        raise ValueError(
            f"Ground truth must have shape (N, 4), "
            f"but got {ground_truth.shape}"
        )

    if len(predictions) != len(ground_truth):
        raise ValueError(
            f"Prediction count ({len(predictions)}) does not match "
            f"ground-truth count ({len(ground_truth)})."
        )

    if len(predictions) == 0:
        raise ValueError("No predictions were provided.")

    iou_scores = np.asarray(
        [
            calculate_iou_xywh(prediction, target)
            for prediction, target in zip(
                predictions,
                ground_truth,
            )
        ],
        dtype=np.float64,
    )

    center_errors = np.asarray(
        [
            calculate_center_error(prediction, target)
            for prediction, target in zip(
                predictions,
                ground_truth,
            )
        ],
        dtype=np.float64,
    )

    # Standard OTB/HOT success-plot thresholds:
    # 0.00, 0.05, ..., 1.00
    overlap_thresholds = np.arange(
        0.0,
        1.0001,
        0.05,
    )

    success_curve = np.asarray(
        [
            np.mean(iou_scores > threshold)
            for threshold in overlap_thresholds
        ],
        dtype=np.float64,
    )

    # Standard tracker AUC calculation: average success
    # rate across all overlap thresholds.
    success_auc = float(np.mean(success_curve))

    # Percentage of frames whose centre error is
    # within 20 pixels.
    distance_precision_20 = float(
        np.mean(center_errors <= 20.0)
    )

    return {
        "success_auc": success_auc,
        "dp_20": distance_precision_20,
        "mean_iou": float(np.mean(iou_scores)),
        "median_iou": float(np.median(iou_scores)),
        "mean_center_error": float(
            np.mean(center_errors)
        ),
        "overlap_thresholds": overlap_thresholds,
        "success_curve": success_curve,
        "iou_scores": iou_scores,
        "center_errors": center_errors,
    }

In [14]:
def main() -> None:
    image_paths = sorted(SEQUENCE_DIR.glob("*.jpg"))
    ground_truth = load_ground_truth(GROUND_TRUTH_PATH)

    if not image_paths:
        raise FileNotFoundError(f"No JPG images found in {SEQUENCE_DIR}")

    if len(image_paths) != len(ground_truth):
        raise ValueError(
            f"Found {len(image_paths)} images but "
            f"{len(ground_truth)} annotation rows."
        )

    first_frame = cv2.imread(str(image_paths[0]))

    if first_frame is None:
        raise RuntimeError(f"Could not read {image_paths[0]}")

    frame_height, frame_width = first_frame.shape[:2]

    # OpenCV 4.13's modern tracker expects built-in integer values:
    # x, y, width, height
    initial_box = tuple(
        int(round(value))
        for value in ground_truth[0]
    )
    
    x, y, width, height = initial_box
    frame_height, frame_width = first_frame.shape[:2]
    
    if width <= 0 or height <= 0:
        raise ValueError(f"Invalid initial box size: {initial_box}")
    
    if x < 0 or y < 0 or x + width > frame_width or y + height > frame_height:
        raise ValueError(
            f"Initial box {initial_box} is outside image dimensions "
            f"{frame_width}x{frame_height}"
        )
    
    print("First frame shape:", first_frame.shape)
    print("Initial box:", initial_box)
    print("Value types:", [type(value) for value in initial_box])
    
    tracker = create_csrt_tracker()
    
    # init() normally returns None on successful initialisation.
    tracker.init(first_frame, initial_box)

    video_writer = cv2.VideoWriter(
        str(OUTPUT_VIDEO_PATH),
        cv2.VideoWriter_fourcc(*"mp4v"),
        20.0,
        (frame_width, frame_height),
    )

    if not video_writer.isOpened():
        raise RuntimeError("Could not create the output video")

    # Frame 1 uses the provided initial box.
    predictions = [initial_box]
    iou_scores = [1.0]

    first_display = first_frame.copy()

    draw_box(
        first_display,
        ground_truth[0],
        (0, 255, 0),
        "Ground truth",
    )
    draw_box(
        first_display,
        initial_box,
        (0, 0, 255),
        "CSRT",
    )

    video_writer.write(first_display)

    for frame_index, image_path in enumerate(image_paths[1:], start=1):
        frame = cv2.imread(str(image_path))

        if frame is None:
            raise RuntimeError(f"Could not read {image_path}")

        tracking_success, predicted_box = tracker.update(frame)

        if tracking_success:
            predicted_box = tuple(map(float, predicted_box))
        else:
            # Simple fallback: retain the previous prediction.
            predicted_box = predictions[-1]

        predictions.append(predicted_box)

        ground_truth_box = ground_truth[frame_index]
        iou = calculate_iou(predicted_box, ground_truth_box)
        iou_scores.append(iou)

        display_frame = frame.copy()

        # Green: ground truth.
        draw_box(
            display_frame,
            ground_truth_box,
            (0, 255, 0),
            "Ground truth",
        )

        # Red: CSRT prediction.
        draw_box(
            display_frame,
            predicted_box,
            (0, 0, 255),
            "CSRT",
        )

        cv2.putText(
            display_frame,
            f"Frame: {frame_index + 1}  IoU: {iou:.3f}",
            (15, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
        )

        if not tracking_success:
            cv2.putText(
                display_frame,
                "Tracking failure",
                (15, 60),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 255),
                2,
            )

        key = cv2.waitKey(1) & 0xFF
        if key == ord("q"):
            print("Display stopped by user. Continuing was cancelled.")
            break

    video_writer.release()
    cv2.destroyAllWindows()

    predictions_array = np.asarray(predictions, dtype=np.float32)

    np.savetxt(
        OUTPUT_PREDICTIONS_PATH,
        predictions_array,
        delimiter="\t",
        fmt="%.0f",
    )

    ground_truth_array = np.asarray(
        ground_truth[:len(predictions_array)],
        dtype=np.float32,
    )
    
    metrics = calculate_competition_metrics(
        predictions=predictions_array,
        ground_truth=ground_truth_array,
    )
    
    print()
    print("=" * 45)
    print("Tracking evaluation")
    print("=" * 45)
    print(f"Evaluated frames: {len(predictions_array)}")
    print(
        f"Success AUC: {metrics['success_auc']:.4f}"
    )
    print(
        f"Distance Precision @20: {metrics['dp_20']:.4f}"
    )
    print(
        f"Distance Precision @20: "
        f"{metrics['dp_20'] * 100:.2f}%"
    )
    print(
        f"Mean IoU: {metrics['mean_iou']:.4f}"
    )
    print(
        f"Median IoU: {metrics['median_iou']:.4f}"
    )
    print(
        f"Mean centre error: "
        f"{metrics['mean_center_error']:.2f} pixels"
    )
    print("=" * 45)
    
    print(f"Predictions saved to: {OUTPUT_PREDICTIONS_PATH}")
    print(f"Visualisation saved to: {OUTPUT_VIDEO_PATH}")

In [15]:
if __name__ == "__main__":
    main()

First frame shape: (272, 512, 3)
Initial box: (210, 32, 99, 134)
Value types: [<class 'int'>, <class 'int'>, <class 'int'>, <class 'int'>]

Tracking evaluation
Evaluated frames: 450
Success AUC: 0.1447
Distance Precision @20: 0.0378
Distance Precision @20: 3.78%
Mean IoU: 0.1199
Median IoU: 0.0549
Mean centre error: 94.01 pixels
Predictions saved to: results/csrt_predictions_cards6.txt
Visualisation saved to: results/csrt_result_cards6.mp4
